# 🧠 The Thinking Budget — GRPO Training Notebook

> Teaching LLMs *when* to reason, not just *how*.

**Theme:** 3.1 World Modeling / Professional Tasks
**Hackathon:** Meta PyTorch OpenEnv 2026 (India)

## What this notebook trains

**Qwen3-1.7B** (thinking-mode) with GRPO on an OpenEnv RL environment whose reward trains **calibrated metacognition** — the agent emits a `<budget_prediction>` before each `<think>` block and is rewarded for calibration, difficulty awareness, and action coupling. Selective reasoning is the only optimum.

CVE triage is the substrate. **Calibrated metacognition** is the contribution.

> **Why 1.7B and not 8B?** The reward shape is model-size-agnostic — same Qwen3 thinking-mode chat template across 1.7B / 4B / 8B. We chose 1.7B because it converges in ~470 GRPO optimizer steps within typical hackathon GPU budgets. To replicate on 4B/8B, set `MODEL_NAME=Qwen/Qwen3-8B` env var below; everything else stays unchanged.

## What you'll see when training is done

- **GRPO reward curve** rising over ~470 optimizer steps
- **Real calibration plot** auto-regenerated from `eval_calibration.json`
  (target: 0.88 diag accuracy, 0.92 P(long | bug))
- **Transfer-domain plot** (held-out non-CVE PR review)
- **Trained LoRA adapter** ready to download

## Requirements

| Setting | Value |
|---|---|
| GPU runtime | **T4 (16GB)** is enough — 1.7B at 4-bit fits with full 4096 context |
| Training time | ~3–4 h on T4, ~1.5–2 h on A10G |
| Storage | ~3 GB (Qwen3-1.7B 4-bit) |

**Before pressing run:** `Runtime → Change runtime type → T4 GPU` (or A10G if available).

> 💡 If you're a hackathon judge: every cell below is reproducible from a fresh Colab. Just run them top-to-bottom.

## 1. Clone the Space (no zip upload needed)

In [ ]:
import os, sys

if not os.path.exists('/content/code-review-env-v3'):
    !git clone https://huggingface.co/spaces/lucid987654/code-review-env-v3 /content/code-review-env-v3

os.chdir('/content/code-review-env-v3')
sys.path.insert(0, os.getcwd())
print(f'cwd: {os.getcwd()}')
print(f'files: {sorted(os.listdir("."))}')

## 2. Install dependencies (3-5 min)

In [ ]:
%%capture install_log
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.17" peft accelerate bitsandbytes
!pip install openenv-core fastmcp datasets matplotlib

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}')
assert torch.cuda.is_available(), 'No GPU detected. Switch runtime to T4/A10G/A100.'

## 3. Verify the environment (sanity check)

Spin up the env, exercise all 6 MCP tools end-to-end. If this fails, training will fail.

In [ ]:
from code_review_env.server.environment import CodeReviewEnvironment
from openenv.core.env_server import CallToolAction
import re

env = CodeReviewEnvironment()
obs = env.reset(seed=42, difficulty='easy')
ctx = obs.metadata['context']
files = re.findall(r'• (.+?)\s+\[', ctx)
print(f'Episode loaded with {len(files)} files\n')

for tool, args in [
    ('read_file',        {'file_path': files[0]}),
    ('search_code',      {'pattern': 'overflow'}),
    ('get_function_list',{'file_path': files[0]}),
    ('flag_vulnerable',  {'file_path': files[0], 'reasoning': 'unchecked memcpy with user-controlled length'}),
    ('skip_file',        {'file_path': files[1] if len(files) > 1 else files[0], 'reasoning': 'header file, declarations only'}),
    ('submit_report',    {'summary': 'Investigation complete. Vulnerability in unchecked buffer copy.', 'confidence': 'medium'}),
]:
    obs = env.step(CallToolAction(tool_name=tool, arguments=args))
    text = str(obs.result.data if hasattr(obs.result, 'data') else obs.result)
    print(f'  {tool:18s} → {text[:80].replace(chr(10), " ")}')

score = re.search(r'TOTAL SCORE: ([\d.]+)', text)
print(f'\nFinal score (untrained baseline): {float(score.group(1)) if score else 0.0:.3f}')

## 4. Run GRPO training (3-7 hours)

Press play, then go write your blog post. Logs stream live.

In [ ]:
%run train_grpo.py

## 5. Render training curves

In [ ]:
from IPython.display import Image, display, Markdown
import json, os

plot_path = 'grpo_output/training_curves.png'
stats_path = 'grpo_output/training_stats.json'

if os.path.exists(plot_path):
    display(Image(plot_path, width=1100))

if os.path.exists(stats_path):
    with open(stats_path) as f:
        s = json.load(f)
    display(Markdown(f"""
### Training Results

| Metric | Value |
|---|---|
| Model | `{s['model']}` |
| Episodes | {s['num_episodes']} |
| Mean reward (early third) | **{s['early_mean']:.3f}** |
| Mean reward (late third) | **{s['late_mean']:.3f}** |
| Improvement | **{s['improvement']:+.3f}** |
| Best reward | {s['max_reward']:.3f} |
"""))

## 6. Before/After comparison (sanity-check the model actually learned)

Reload the trained LoRA adapter and run it on the same scenario as Cell 3. If training worked, the trained model should produce a higher score than the untrained baseline.

In [ ]:
# Load trained model and run a deterministic eval on 5 episodes
import os, sys, re
sys.path.insert(0, os.getcwd())

if os.path.exists('eval_baseline.py'):
    %run eval_baseline.py
else:
    print('eval_baseline.py not found - skipping. The training_curves.png already shows the improvement curve.')

In [ ]:
## 6b. Regenerate the hero "Thinking Budget" plot with whatever traces we have
# This produces grpo_output/thinking_allocation.png — the README's hero image.
# If eval_baseline.py wrote eval_traces.json, this uses real model traces;
# otherwise it falls back to the heuristic-proxy pattern.
import os, sys
if not os.path.exists('scripts/generate_thinking_viz.py'):
    print('scripts/generate_thinking_viz.py not present in this Space copy — skipping.')
else:
    mode = 'real' if os.path.exists('grpo_output/eval_traces.json') else 'heuristic'
    print(f'Generating hero plot in mode={mode}')
    %run scripts/generate_thinking_viz.py --mode $mode
    from IPython.display import Image, display
    if os.path.exists('grpo_output/thinking_allocation.png'):
        display(Image('grpo_output/thinking_allocation.png', width=1100))

## 7. Download trained adapter + plots

In [ ]:
import shutil

shutil.make_archive('/content/grpo_results', 'zip', './grpo_output')
print('Zipped grpo_output/ -> /content/grpo_results.zip')

try:
    from google.colab import files
    files.download('/content/grpo_results.zip')
except ImportError:
    print('(not in Colab; zip is at /content/grpo_results.zip)')